# 旧パイプライン（コードレビュー前）でのラウンド感度分析

コードレビューで加えた変更（I2: bfill削除、I4: Absolute_Meeting_ID<8除外）を
**元に戻した**パイプラインで rounds=[100,200,300,500,700,1000] を比較する。

目的：旧コードでのIC（≈0.39/0.63）が再現できるかを確認し、
その上で最適ラウンド数を特定する。

In [ ]:
import sys
sys.path.insert(0, '..')

import bisect
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

from src.features import generate_features
from src.modeling import walk_forward_validation

EXCEL_PATH   = '../data/BOJ_data.xlsx'
MEETING_PATH = '../data/BOJ_meeting_history.csv'
START_DATE   = '2024-01-01'

In [ ]:
# ===== 旧 processing.py の load_and_clean_data =====
# 変更点: bfill() を復元（I2リバート）

def load_and_clean_data_OLD(excel_path, meeting_csv_path):
    df_raw = pd.read_excel(excel_path)
    df = df_raw.iloc[1:].copy()
    df['日付'] = pd.to_datetime(df['日付'], format='%Y年%m月%d日')
    df = df.sort_values('日付').reset_index(drop=True)

    rename_dict = {
        'JPBOJ1ONI=TRDT (MID_PRICE)': 'M1',
        'JPBOJ2ONI=TRDT (MID_PRICE)': 'M2',
        'JPBOJ3ONI=TRDT (MID_PRICE)': 'M3',
        'JPBOJ4ONI=TRDT (MID_PRICE)': 'M4',
        'JPBOJ5ONI=TRDT (MID_PRICE)': 'M5',
        'JPBOJ6ONI=TRDT (MID_PRICE)': 'M6',
        'JPBOJ7ONI=TRDT (MID_PRICE)': 'M7',
        'JPBOJ8ONI=TRDT (MID_PRICE)': 'M8',
        'JPY= (MID_PRICE)': 'USDJPY',
        'JGBc1 (TRDPRC_1)': 'JGB_Future',
        '.N225 (TRDPRC_1)': 'Nikkei225',
        '.DXY (TRDPRC_1)': 'DXY',
        'JP12MONI=TRDT (BID)': 'T12',
        'JP18MONI=TRDT (BID)': 'T18',
        'JP24MONI=TRDT (BID)': 'T24',
    }
    df = df.rename(columns=rename_dict)

    cols_to_keep = ['日付', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8',
                    'USDJPY', 'JGB_Future', 'Nikkei225', 'DXY', 'T12', 'T18', 'T24']
    df = df[[c for c in cols_to_keep if c in df.columns]].copy()

    numeric_cols = df.columns.drop('日付')
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df_meetings = pd.read_csv(meeting_csv_path)
    df_meetings['Date'] = pd.to_datetime(df_meetings['Date'])
    df = pd.merge(df, df_meetings[['Date', 'Policy_Rate', 'Event']],
                  left_on='日付', right_on='Date', how='left', suffixes=('', '_mtg'))
    df = df.drop(columns=['Date']).rename(columns={'日付': 'Date'})
    df['Is_Meeting_Day'] = df['Event'].notnull().astype(int)

    # ★ I2リバート: bfill() を復元
    df['Actual_Policy_Rate'] = df['Policy_Rate'].ffill().bfill()

    all_rate_cols = ['M1','M2','M3','M4','M5','M6','M7','M8','T12','T18','T24']
    all_rate_cols = [c for c in all_rate_cols if c in df.columns]
    for col in all_rate_cols:
        df[f'{col}_is_imputed'] = df[col].isnull().astype(int)

    impute_target_cols = all_rate_cols + ['USDJPY', 'JGB_Future', 'Nikkei225', 'DXY']
    impute_target_cols = [c for c in impute_target_cols if c in df.columns]
    imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=20, random_state=42)
    df[impute_target_cols] = imputer.fit_transform(df[impute_target_cols])

    all_meeting_dates = sorted(df_meetings['Date'].unique())
    def days_to_next_mpm(date):
        idx = bisect.bisect_left(all_meeting_dates, date)
        if idx < len(all_meeting_dates):
            return (all_meeting_dates[idx] - date).days
        return np.nan
    date_to_days = {d: days_to_next_mpm(d) for d in df['Date'].unique()}
    df['Days_to_MPM'] = df['Date'].map(date_to_days)

    tenor_cols = [c for c in ['T12', 'T18', 'T24'] if c in df.columns]
    ext_cols = [c for c in ['USDJPY', 'JGB_Future', 'Nikkei225', 'DXY'] if c in df.columns]
    final_cols = (['Date', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8'] +
                  tenor_cols + ext_cols +
                  ['Actual_Policy_Rate', 'Is_Meeting_Day', 'Days_to_MPM'] +
                  [f'{c}_is_imputed' for c in all_rate_cols])
    return df[final_cols]

In [ ]:
# ===== 旧 pooling.py の pool_boj_data =====
# 変更点: I4ブロック（Absolute_Meeting_ID<8除外）を削除

def pool_boj_data_OLD(df):
    boj_rate_cols   = [f'M{i}' for i in range(1, 9)]
    tenor_rate_cols = [c for c in ['T12', 'T18', 'T24'] if c in df.columns]
    raw_rate_cols   = boj_rate_cols + tenor_rate_cols
    id_cols = [col for col in df.columns if col not in raw_rate_cols]

    pooled_boj = df.melt(id_vars=id_cols, value_vars=boj_rate_cols,
                         var_name='Rate_Label', value_name='Rate_Value')
    pooled_boj['Meeting_Index'] = pooled_boj['Rate_Label'].str.extract(r'(\d+)').astype(int)
    pooled_boj['Is_Tenor_OIS']  = 0

    tenor_index_map = {'T12': 10, 'T18': 11, 'T24': 12}
    if tenor_rate_cols:
        pooled_tenor = df.melt(id_vars=id_cols, value_vars=tenor_rate_cols,
                               var_name='Rate_Label', value_name='Rate_Value')
        pooled_tenor['Meeting_Index'] = pooled_tenor['Rate_Label'].map(tenor_index_map)
        pooled_tenor['Is_Tenor_OIS']  = 1
        pooled = pd.concat([pooled_boj, pooled_tenor], ignore_index=True)
    else:
        pooled = pooled_boj

    pooled = pooled.sort_values(['Rate_Label', 'Date']).reset_index(drop=True)

    df_dates = df[['Date', 'Days_to_MPM']].drop_duplicates()
    df_dates = df_dates.dropna(subset=['Days_to_MPM'])
    df_dates['Next_Meeting_Date'] = df_dates['Date'] + pd.to_timedelta(
        df_dates['Days_to_MPM'].astype(int), unit='D')
    all_meeting_dates = sorted(df_dates['Next_Meeting_Date'].unique())
    meeting_date_to_rank = {m: i for i, m in enumerate(all_meeting_dates)}
    date_to_next_rank = dict(zip(df_dates['Date'], df_dates['Next_Meeting_Date'].map(meeting_date_to_rank)))

    pooled['_next_rank'] = pooled['Date'].map(date_to_next_rank)
    pooled['Absolute_Meeting_ID'] = (pooled['_next_rank'] + pooled['Meeting_Index'] - 1).astype('Int64')
    pooled.loc[pooled['Is_Tenor_OIS'] == 1, 'Absolute_Meeting_ID'] = pd.NA
    pooled = pooled.drop(columns=['_next_rank'])

    data_start_date = df['Date'].min()
    def get_first_seen_date(abs_id):
        if pd.isna(abs_id): return pd.NaT
        k = int(abs_id)
        predecessor_idx = k - 8
        if predecessor_idx < 0: return data_start_date
        elif predecessor_idx < len(all_meeting_dates): return all_meeting_dates[predecessor_idx]
        else: return pd.NaT

    abs_id_to_first_seen = {k: get_first_seen_date(k) for k in pooled['Absolute_Meeting_ID'].dropna().unique()}
    pooled['First_Seen_Date'] = pooled['Absolute_Meeting_ID'].map(abs_id_to_first_seen)
    pooled['Days_since_first_seen'] = (pooled['Date'] - pooled['First_Seen_Date']).dt.days
    pooled = pooled.drop(columns=['First_Seen_Date'])

    # ★ I4リバート: Absolute_Meeting_ID<8 の除外ブロックを削除

    for h in [1, 3, 5]:
        pooled[f'Target_{h}d'] = (
            pooled.groupby('Rate_Label')['Rate_Value'].shift(-h) - pooled['Rate_Value']
        )

    for h in [3, 5]:
        instr_std = pooled.groupby('Rate_Label')[f'Target_{h}d'].transform('std')
        pooled[f'Target_{h}d_std']  = instr_std
        pooled[f'Target_{h}d_norm'] = pooled[f'Target_{h}d'] / instr_std

    df_date_meeting = df[['Date', 'Is_Meeting_Day']].copy()
    df_date_meeting['is_post_mpm'] = (
        df_date_meeting['Is_Meeting_Day'].rolling(window=6, min_periods=1).max() == 1
    ).astype(int)
    pooled = pd.merge(pooled, df_date_meeting[['Date', 'is_post_mpm']], on='Date', how='left')

    boj_spread_cols   = [f'M{i}_spread'     for i in range(1, 9)]
    boj_fd_cols       = [f'M{i}_frac_diff'  for i in range(1, 9)]
    boj_imp_cols      = [f'M{i}_is_imputed' for i in range(1, 9)]
    tenor_spread_cols = [f'{c}_spread'     for c in tenor_rate_cols]
    tenor_fd_cols     = [f'{c}_frac_diff'  for c in tenor_rate_cols]
    tenor_imp_cols    = [f'{c}_is_imputed' for c in tenor_rate_cols]
    ext_fd_cols       = ['USDJPY_frac_diff', 'JGB_Future_frac_diff', 'Nikkei225_frac_diff']
    basic_cols  = ['Date', 'Rate_Label', 'Meeting_Index', 'Is_Tenor_OIS',
                   'is_post_mpm', 'Days_to_MPM', 'Actual_Policy_Rate',
                   'Absolute_Meeting_ID', 'Days_since_first_seen']
    target_cols = ['Target_1d',
                   'Target_3d', 'Target_3d_norm', 'Target_3d_std',
                   'Target_5d', 'Target_5d_norm', 'Target_5d_std']
    curve_cols    = ['Curve_Slope'] + [f'Butterfly_M{n}' for n in range(2, 8)]
    curve_fd_cols = [f'{c}_frac_diff' for c in curve_cols]
    cyclic_cols   = ['DayOfWeek_sin', 'DayOfWeek_cos', 'Days_to_MPM_sin', 'Days_to_MPM_cos']
    final_cols = (basic_cols + boj_spread_cols + boj_fd_cols + boj_imp_cols
                  + tenor_spread_cols + tenor_fd_cols + tenor_imp_cols
                  + ext_fd_cols + curve_cols + curve_fd_cols + cyclic_cols + target_cols)
    available_cols = [c for c in final_cols if c in pooled.columns]
    return pooled[available_cols]

In [ ]:
# 旧パイプラインでデータ準備
df_raw_old    = load_and_clean_data_OLD(EXCEL_PATH, MEETING_PATH)
df_feat_old   = generate_features(df_raw_old)
df_pooled_old = pool_boj_data_OLD(df_feat_old)

print(f'旧パイプライン データ準備完了: {len(df_pooled_old):,} 行')

In [ ]:
# 旧パイプラインでラウンド感度分析
ROUNDS_LIST = [100, 200, 300, 500, 700, 1000]
records = []

for rounds in ROUNDS_LIST:
    print(f'rounds={rounds} を実行中...')

    res_3d = walk_forward_validation(df_pooled_old, 'Target_3d_norm', START_DATE, num_boost_round=rounds)
    res_5d = walk_forward_validation(df_pooled_old, 'Target_5d_norm', START_DATE, num_boost_round=rounds)

    ic_3d, _ = spearmanr(res_3d['Actual'], res_3d['Pred'])
    ic_5d, _ = spearmanr(res_5d['Actual'], res_5d['Pred'])
    train_ic_3d = res_3d['Train_IC'].mean()
    train_ic_5d = res_5d['Train_IC'].mean()

    records.append({
        'rounds':      rounds,
        'OOS_IC_3d':   round(ic_3d, 4),
        'OOS_IC_5d':   round(ic_5d, 4),
        'Train_IC_3d': round(train_ic_3d, 4),
        'Train_IC_5d': round(train_ic_5d, 4),
        'Gap_3d':      round(train_ic_3d - ic_3d, 4),
        'Gap_5d':      round(train_ic_5d - ic_5d, 4),
    })

df_old = pd.DataFrame(records)
print('\n=== 旧パイプライン 感度分析結果 ===')
print(df_old.to_string(index=False))

In [ ]:
# 参照用: 現行パイプライン結果（rounds_sensitivity.ipynb から転記）
current_data = {
    'rounds':      [100,   200,   300,   500,   700,   1000],
    'OOS_IC_3d':   [0.1984,0.1845,0.1814,0.1792,0.1805,0.1802],
    'OOS_IC_5d':   [0.1914,0.1920,0.1908,0.1882,0.1893,0.1881],
}
df_current = pd.DataFrame(current_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col_3d, col_5d, title in [
    (axes[0], 'OOS_IC_3d', 'OOS_IC_3d', '3d モデル OOS IC'),
    (axes[1], 'OOS_IC_5d', 'OOS_IC_5d', '5d モデル OOS IC'),
]:
    key = col_3d if '3d' in col_3d else col_5d
    ax.plot(df_old['rounds'],     df_old[key],     'o-', label='旧パイプライン', color='steelblue', linewidth=2)
    ax.plot(df_current['rounds'], df_current[key], 's--', label='現行パイプライン', color='tomato',    linewidth=2)
    ax.axvline(300, color='gray', linestyle=':', linewidth=1, label='現行設定(300)')
    ax.set_title(title)
    ax.set_xlabel('num_boost_round')
    ax.set_ylabel('OOS IC (Spearman)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../designs/pending/old_vs_new_pipeline_rounds.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n=== 300ラウンド時点での比較 ===')
r300_old = df_old[df_old['rounds']==300].iloc[0]
r300_cur = df_current[df_current['rounds']==300].iloc[0]
print(f'旧パイプライン: 3d={r300_old["OOS_IC_3d"]:.4f}, 5d={r300_old["OOS_IC_5d"]:.4f}')
print(f'現行パイプライン: 3d={r300_cur["OOS_IC_3d"]:.4f}, 5d={r300_cur["OOS_IC_5d"]:.4f}')